# Video to UX Narrative Pipeline
This notebook runs the lightning-fast, zero-data-loss extraction pipeline and sends the frames to Gemini 3.5 Flash.


In [ ]:
import os
import cv2
import numpy as np
import time
import json
from PIL import Image
import google.generativeai as genai
from dotenv import load_dotenv
from IPython.display import display

# Setup Gemini
load_dotenv()
genai.configure(api_key=os.getenv('GEMINI_API_KEY'))
vision_model = genai.GenerativeModel('gemini-3.5-flash')


In [ ]:
def is_blurry(image: np.ndarray, threshold: float = 100.0) -> bool:
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    variance = cv2.Laplacian(gray, cv2.CV_64F).var()
    return variance < threshold

def extract_keyframes(video_path: str, diff_threshold: float = 0.01):
    cap = cv2.VideoCapture(video_path)
    frames = []
    
    ret, prev_frame = cap.read()
    if not ret:
        cap.release()
        return frames
        
    frames.append(prev_frame)
    
    h, w = prev_frame.shape[:2]
    math_w = 240
    math_h = int((math_w / float(w)) * h)
    
    prev_math_img = cv2.resize(prev_frame, (math_w, math_h))
    prev_gray = cv2.cvtColor(prev_math_img, cv2.COLOR_BGR2GRAY)
    
    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps <= 0: fps = 30
    step = max(1, int(fps / 5)) 
    
    frame_count = 1
    while True:
        ret = cap.grab()
        if not ret: break
            
        if frame_count % step == 0:
            ret, curr_frame = cap.retrieve()
            if not ret: continue
                
            curr_math_img = cv2.resize(curr_frame, (math_w, math_h))
            curr_gray = cv2.cvtColor(curr_math_img, cv2.COLOR_BGR2GRAY)
            
            diff = cv2.absdiff(curr_gray, prev_gray)
            _, thresh = cv2.threshold(diff, 30, 255, cv2.THRESH_BINARY)
            changed_pixels = cv2.countNonZero(thresh)
            total_pixels = math_w * math_h
            
            if (changed_pixels / total_pixels) > diff_threshold:
                frames.append(curr_frame)
                prev_gray = curr_gray
                
        frame_count += 1
            
    cap.release()
    if len(frames) > 60:
        step_down = len(frames) / 60.0
        return [frames[int(i * step_down)] for i in range(60)]
    return frames


In [ ]:
video_file = "front_product.mp4" # Replace with your video path

print(f"Extracting keyframes from {video_file}...")
start_time = time.time()
keyframes = extract_keyframes(video_file)
print(f"Extraction took {time.time() - start_time:.2f}s. Total frames: {len(keyframes)}")

clear_pil_images = []
for frame in keyframes:
    h, w = frame.shape[:2]
    max_dim = 768.0
    if max(h, w) > max_dim:
        scale = max_dim / max(h, w)
        payload_frame = cv2.resize(frame, (int(w * scale), int(h * scale)))
    else:
        payload_frame = frame
        
    if not is_blurry(payload_frame):
        rgb_image = cv2.cvtColor(payload_frame, cv2.COLOR_BGR2RGB)
        clear_pil_images.append(Image.fromarray(rgb_image))

print(f"Kept {len(clear_pil_images)} clear frames.")

# Display the first 5 frames
for img in clear_pil_images[:5]:
    display(img)


In [ ]:
prompt = '''
You are an expert Product Manager and UX Researcher.
I am providing you with a chronological sequence of frames extracted from a UI screen recording video.

CRITICAL INSTRUCTIONS:
1. DO NOT HALLUCINATE. You must ONLY describe exactly what is visibly present in the frames. Do not guess, assume, or invent any actions, text, or context that is not explicitly shown in the pixels.
2. BE HIGHLY EXPLAINABLE AND DETAILED. Extract as much concrete information as possible. Read the exact text on the screen, describe the exact layout of the dashboard, and specify the exact names of menus, buttons, charts, or search queries visible.
3. Your narrative must be a step-by-step, highly detailed breakdown of the user's journey, explaining what they saw and what they did based STRICTLY on the visual evidence.

Please provide your output in valid JSON format exactly like this:
{
  "scene_captions": [
    "Frame 1: The user is on a dashboard titled 'X'. The sidebar contains menus for 'Y' and 'Z'. A bar chart is visible showing...",
    "Frame 2: The user clicks the 'Settings' button located in the top right..."
  ],
  "narrative": "A cohesive, highly detailed user story narrative weaving the chronological actions together. Must be strictly grounded in the visual evidence."
}
'''

print("Sending to Gemini 3.5 Flash...")
start_time = time.time()
contents = [prompt] + clear_pil_images
response = vision_model.generate_content(contents)
print(f"Gemini processing took {time.time() - start_time:.2f}s")

# Parse and print JSON nicely
try:
    result = json.loads(response.text.replace("```json", "").replace("```", "").strip())
    print(json.dumps(result, indent=2))
except Exception as e:
    print("Raw Response:")
    print(response.text)
